# Common functions

Contains shared helper functions used across the bronze and silver layers.


In [0]:
### Imports
from pyspark.sql import functions as F
from pyspark.sql import types as T
import re

## Bronze functions
These functions support ingestion tasks such as column name standardization and technical metadata enrichment.
- standardize_column_name
- standardize_columns
- add_ingestion_metadata

In [0]:
def standardize_column_name(col_name: str) -> str:
    """Convert one column name into lowercase snake_case."""
    col_name = col_name.strip().lower()
    col_name = re.sub(r"[^a-z0-9]+", "_", col_name)
    col_name = re.sub(r"_+", "_", col_name)
    return col_name.strip("_")

def standardize_columns(df):
    """Apply standardized snake_case naming to all dataframe columns."""
    return df.toDF(*[standardize_column_name(c) for c in df.columns])

def add_ingestion_metadata(df, source_name: str, source_file: str):
    """Add source and ingestion timestamp metadata columns."""
    return (
        df.withColumn("ingestion_timestamp", F.current_timestamp())
          .withColumn("source_name", F.lit(source_name))
          .withColumn("source_file", F.lit(source_file))
    )

## Silver functions
These functions support light cleaning, typing, normalization, and structural adjustments in the silver layer.

- replace_values_with_null
- replace_negative_with_null
- trim_upper_columns
- parse_yyyymmdd_columns
- yn_to_boolean
- convert_yn_columns_to_boolean
- cast_columns
- rename_columns
- reorder_columns
- drop_duplicates_by_key
- parse_custom_datetime_to_date
- parse_custom_datetime_columns

In [0]:
def replace_values_with_null(df, columns: list, invalid_values: list | None = None):
    """
    Replace selected placeholder values with null in the given columns.

    If invalid_values is not provided, a default list of NA-like string values is used.
    String columns are normalized with trim + upper before comparison.
    Numeric columns are compared directly.
    """
    default_na_like_values = [
        "",
        "N/A",
        "NULL",
        "NONE",
        "UNKNOWN",
        "NOT AVAILABLE",
        "NOT APPLICABLE",
        "UNSPECIFIED",
        "-"
    ]

    invalid_values = invalid_values if invalid_values is not None else default_na_like_values

    schema_dict = {field.name: field.dataType for field in df.schema.fields}

    for c in columns:
        if c not in df.columns:
            continue

        col_type = schema_dict[c]

        if isinstance(col_type, T.StringType):
            normalized_string_values = [
                str(v).strip().upper()
                for v in invalid_values
                if isinstance(v, str)
            ]

            df = df.withColumn(
                c,
                F.when(
                    F.upper(F.trim(F.col(c))).isin(normalized_string_values),
                    None
                ).otherwise(F.col(c))
            )
        else:
            non_string_values = [
                v for v in invalid_values
                if not isinstance(v, str)
            ]

            if non_string_values:
                df = df.withColumn(
                    c,
                    F.when(F.col(c).isin(non_string_values), None).otherwise(F.col(c))
                )

    return df

def replace_negative_with_null(df, columns: list):
    """Replace negative numeric values with null in the selected columns."""
    for c in columns:
        if c in df.columns:
            df = df.withColumn(
                c,
                F.when(F.col(c) < 0, None).otherwise(F.col(c))
            )
    return df

def trim_upper_columns(df, columns: list):
    """Trim spaces and convert selected columns to uppercase."""
    for c in columns:
        if c in df.columns:
            df = df.withColumn(c, F.upper(F.trim(F.col(c))))
    return df

def parse_yyyymmdd_columns(df, date_columns_map: dict):
    """Parse multiple yyyyMMdd columns into date columns using a mapping."""
    for source_col, target_col in date_columns_map.items():
        if source_col in df.columns:
            df = df.withColumn(
                target_col,
                F.to_date(
                    F.lpad(F.col(source_col).cast("string"), 8, "0"),
                    "yyyyMMdd"
                )
            )
    return df

def yn_to_boolean(col_name: str):
    """Return a boolean expression from a Y/N source column."""
    return (
        F.when(F.upper(F.trim(F.col(col_name))) == "Y", F.lit(True))
         .when(F.upper(F.trim(F.col(col_name))) == "N", F.lit(False))
         .otherwise(F.lit(None).cast("boolean"))
    )


def convert_yn_columns_to_boolean(df, columns: list):
    """Convert multiple Y/N columns into boolean columns."""
    for c in columns:
        if c in df.columns:
            df = df.withColumn(c, yn_to_boolean(c))
    return df


def cast_columns(df, cast_map: dict):
    """Cast multiple columns using a dictionary of target data types."""
    for c, target_type in cast_map.items():
        if c in df.columns:
            df = df.withColumn(c, F.col(c).cast(target_type))
    return df


def rename_columns(df, rename_map: dict):
    """Rename multiple columns using a source-to-target mapping."""
    for old_name, new_name in rename_map.items():
        if old_name in df.columns and old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)
    return df


def reorder_columns(df, preferred_order: list):
    """Move selected columns to the front and keep the remaining columns after them."""
    final_columns = [c for c in preferred_order if c in df.columns] + [
        c for c in df.columns if c not in preferred_order
    ]
    return df.select(*final_columns)


def drop_duplicates_by_key(df, key_columns: list):
    """Drop duplicate rows using the selected key columns."""
    return df.dropDuplicates(key_columns)

def parse_custom_datetime_to_date(col_name: str):
    """Convert a datetime string like 'Tue Jan 01 00:00:00 EST 2013' into a date expression."""
    month_abbr = F.regexp_extract(F.col(col_name), r"^[A-Za-z]{3}\s([A-Za-z]{3})", 1)

    month_num = (
        F.when(month_abbr == "Jan", "01")
         .when(month_abbr == "Feb", "02")
         .when(month_abbr == "Mar", "03")
         .when(month_abbr == "Apr", "04")
         .when(month_abbr == "May", "05")
         .when(month_abbr == "Jun", "06")
         .when(month_abbr == "Jul", "07")
         .when(month_abbr == "Aug", "08")
         .when(month_abbr == "Sep", "09")
         .when(month_abbr == "Oct", "10")
         .when(month_abbr == "Nov", "11")
         .when(month_abbr == "Dec", "12")
    )

    year_part = F.regexp_extract(F.col(col_name), r"(\d{4})$", 1)
    day_part = F.regexp_extract(F.col(col_name), r"^[A-Za-z]{3}\s[A-Za-z]{3}\s(\d{2})", 1)

    return F.to_date(
        F.concat_ws("-", year_part, month_num, day_part),
        "yyyy-MM-dd"
    )

def parse_custom_datetime_columns(df, columns: list):
    """Convert multiple datetime string columns like 'Tue Jan 01 00:00:00 EST 2013' into date columns."""
    for c in columns:
        if c in df.columns:
            df = df.withColumn(c, parse_custom_datetime_to_date(c))
    return df

